In [1]:
import pandas as pd

RECORDS = "records.csv"

records = pd.read_csv(RECORDS)

# 1) Sum delta_H and kl over positions, for each (task, ratio).
per_task = (
    records.groupby(["ratio", "task_id"], as_index=False)[["delta_H", "kl"]]
    .sum()
    .rename(columns={"delta_H": "delta_H_sum", "kl": "kl_sum"})
)

# 2) Average those per-task sums over tasks, for each ratio.
per_ratio = (
    per_task.groupby("ratio")
    .agg(
        delta_H_mean=("delta_H_sum", "mean"),
        delta_H_std=("delta_H_sum", "std"),
        kl_mean=("kl_sum", "mean"),
        kl_std=("kl_sum", "std"),
        n_tasks=("task_id", "nunique"),
    )
    .reset_index()
)

per_ratio

,ratio,delta_H_mean,delta_H_std,kl_mean,kl_std,n_tasks
0,0.25,0.724721,1.988847,1.087761,3.305014,150
1,0.50,1.493464,2.428354,1.829862,3.967871,150
2,0.75,2.885545,3.886925,4.810048,7.840003,150
3,0.95,4.527524,5.321688,11.482278,10.882397,150


In [ ]:
import matplotlib.pyplot as plt

# Two panels rather than one axes with two y scales: delta_H and KL are different
# quantities on different ranges, and a twin axis would let the choice of scales imply
# a crossover that isn't in the data. Same style as `_draw_summary_panel` in
# evaluation/entropy_plots.py, so these read the same as the generated figures.
PANELS = [
    ("delta_H_mean", "delta_H_std", r"$\sum_t \Delta H_t$  (bits)", "Information gain"),
    ("kl_mean", "kl_std", r"$\sum_t \mathrm{KL}(p_{full} \| p_{comp})_t$  (bits)", "Distribution shift"),
]

n_tasks = int(per_ratio["n_tasks"].max())
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, constrained_layout=True)

for ax, (mean_col, std_col, ylabel, title) in zip(axes, PANELS):
    ax.errorbar(
        per_ratio["ratio"], per_ratio[mean_col], yerr=per_ratio[std_col],
        marker="o", markersize=6, capsize=3, linewidth=2,
    )
    # Zero is a real reference for both: delta_H is a signed difference, and KL cannot
    # legitimately go below it.
    ax.axhline(0, color="black", linewidth=0.5)
    ax.grid(alpha=0.25, linewidth=0.5)
    ax.set_axisbelow(True)
    ax.set_xticks(per_ratio["ratio"])
    ax.set_xlabel("Compression ratio")
    ax.set_ylabel(ylabel)
    ax.set_title(title)

fig.suptitle(f"Distortion vs compression ratio  (mean ± std over {n_tasks} tasks)")
plt.show()